In [43]:
import cv2
import mediapipe as mp
import time
import numpy as np

def calculate_angle(a, b, c):
    
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360.0 - angle
        
    return angle

def classify_pose(landmarks):
   
    shoulder = [landmarks[11].x, landmarks[11].y] # left shoulder
    hip = [landmarks[23].x, landmarks[23].y]       # left hip
    knee = [landmarks[25].x, landmarks[25].y]      # left knee
    ankle = [landmarks[27].x, landmarks[27].y]     # left ankle

    # checking if lays
    body_y_dist = abs(shoulder[1] - hip[1])
    body_x_dist = abs(shoulder[0] - hip[0])
    
    if body_x_dist > body_y_dist and body_y_dist < 0.15:
        return "lays"

    # angles to check if sits/stands
    hip_angle = calculate_angle(shoulder, hip, knee)
    knee_angle = calculate_angle(hip, knee, ankle)

    if hip_angle < 130 and knee_angle < 130:
        return "sits"
        
    elif hip_angle > 150 and knee_angle > 150:
        return "stands"
    else:
        return "undefined"



model_path = 'mediapipe_weights/pose_landmarker_full.task'

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    min_pose_detection_confidence=0.7,
    min_pose_presence_confidence=0.7,
    min_tracking_confidence=0.7,
    num_poses = 1
)


BODY_CONNECTIONS = [
    #(9, 10), (8, 6), (6, 5), (5, 4), (4, 0), (0, 1), (1, 2), (2, 3), (3, 7), # face
    (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19), # left part
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),  # right part
    (12, 24), (24, 26), (26, 28), (28, 30), (30, 32), (32, 28), # right leg
    (11, 23), (23, 25), (25, 27), (27, 29), (29, 31), (31, 27) # left leg
]

cap = cv2.VideoCapture('videos/fall/2.mp4')
WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(WIDTH, HEIGHT)

with PoseLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
            
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        timestamp_ms = int(time.time() * 1000)

        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        h, w, _ = frame.shape
        if result.pose_landmarks:
            for landmarks in result.pose_landmarks:
                pose = classify_pose(landmarks)

                for connection in BODY_CONNECTIONS:
                    start_idx, end_idx = connection
                    pt1 = landmarks[start_idx]
                    pt2 = landmarks[end_idx]
                    
                    if pt1.visibility > 0.5 and pt2.visibility > 0.5:
                        cx1, cy1 = int(pt1.x * w), int(pt1.y * h)
                        cx2, cy2 = int(pt2.x * w), int(pt2.y * h)
                        cv2.line(frame, (cx1, cy1), (cx2, cy2), (0, 255, 0), 2)
        
                for idx in range(min(BODY_CONNECTIONS)[0], max(BODY_CONNECTIONS)[1]):
                    landmark = landmarks[idx]
                    if landmark.visibility > 0.5:
                        cx, cy = int(landmark.x * w), int(landmark.y * h)
                        
                        color = (255, 0, 0) if idx % 2 == 1 else (0, 0, 255) # blue - left, red - right
                        cv2.circle(frame, (cx, cy), 5, color, -1)
                        # if cap.getBackendName() == 'MSMF':
                        #     frame = cv2.flip(frame, 1)
                        #     cv2.putText(frame, str(idx), (WIDTH-cx+10, cy+10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 1, cv2.LINE_AA)
                        #     frame = cv2.flip(frame, 1)
                        # else:
                        #     cv2.putText(frame, str(idx), (cx+10, cy+10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 1, cv2.LINE_AA)

        flipped_frame = cv2.flip(frame, 1) if cap.getBackendName() == 'MSMF' else frame
        flipped_frame = cv2.resize(frame, (540, 960))
        cv2.imshow('Mediapipe_pose', flipped_frame)
        cv2.setWindowTitle("Mediapipe_pose", f"Mediapipe --- STATE {pose}")


        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


1080 1920


In [13]:
import cv2
import mediapipe as mp
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os

# --- 1. АРХИТЕКТУРА LSTM МОДЕЛИ ---
class FallDetectionLSTM(nn.Module):
    def __init__(self, input_size=66, hidden_size=64, num_layers=2, num_classes=2):
        super(FallDetectionLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM слой обрабатывает последовательность кадров
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        # Полносвязный слой для классификации состояния на текущем/последнем кадре
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        # Берем выход последнего временного шага для классификации
        out = self.fc(out[:, -1, :])
        return out

# --- 2. КАСТОМНЫЙ ДАТАСЕТ ДЛЯ ПРЕДОБРАБОТКИ ВИДЕО ---
class FallVideoDataset(Dataset):
    def __init__(self, video_dir, num_videos=13, seq_length=30, fall_threshold_ratio=0.6):
        self.seq_length = seq_length
        raw_data = []
        raw_labels = []
        
        # Настройка MediaPipe
        model_path = 'mediapipe_weights/pose_landmarker_full.task'
        options = mp.tasks.vision.PoseLandmarkerOptions(
            base_options=mp.tasks.BaseOptions(model_asset_path=model_path),
            running_mode=mp.tasks.vision.RunningMode.VIDEO,
            min_pose_detection_confidence=0.5,
            num_poses=1
        )
        
        print("-> Начало извлечения ключевых точек из видео...")
        
        for i in range(1, num_videos + 1):
            video_path = os.path.join(video_dir, f"{i}.mp4")
            if not os.path.exists(video_path):
                print(f"Файл не найден: {video_path}, пропускаем.")
                continue
                
            cap = cv2.VideoCapture(video_path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            if fps <= 0: 
                fps = 30.0
                
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            fall_start_frame = int(total_frames * fall_threshold_ratio)
            
            landmarks_sequence = []
            frame_idx = 0
            
            with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
                while cap.isOpened():
                    success, frame = cap.read()
                    if not success:
                        break
                        
                    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                    timestamp_ms = int(frame_idx * (1000 / fps))
                    
                    try:
                        result = landmarker.detect_for_video(mp_image, timestamp_ms)
                        
                        # Вектор признаков текущего кадра
                        frame_features = []
                        if result.pose_landmarks and len(result.pose_landmarks) > 0:
                            # Извлекаем x, y для всех 33 точек первой найденной позы
                            for lm in result.pose_landmarks[0]:
                                frame_features.extend([lm.x, lm.y])
                        
                        # Если точки не найдены или их меньше 33, заполняем нулями до 66 признаков
                        if len(frame_features) < 66:
                            frame_features = [0.0] * 66
                            
                        label = 1 if frame_idx >= fall_start_frame else 0
                        landmarks_sequence.append((frame_features, label))
                        
                    except Exception as e:
                        print(f"Ошибка на кадре {frame_idx} в видео {i}: {e}")
                        
                    frame_idx += 1
                    
            cap.release()
            
            # Нарезаем кадры ТЕКУЩЕГО видео на скользящие окна
            for idx in range(len(landmarks_sequence) - seq_length + 1):
                window = landmarks_sequence[idx : idx + seq_length]
                
                # Разделяем признаки и метки внутри окна
                window_features = [item[0] for item in window] # Список из 30 векторов по 66 элементов
                window_label = window[-1][1]                   # Метка последнего кадра в окне
                
                raw_data.append(window_features)
                raw_labels.append(window_label)
                
        if len(raw_data) == 0:
            raise RuntimeError(f"Не удалось извлечь данные. Проверьте путь '{video_dir}' и корректность видеофайлов.")
            
        print(f"-> Извлечение завершено. Создано {len(raw_data)} тренировочных последовательностей.")
        
        # Преобразуем в тензоры PyTorch и сохраняем как атрибуты класса
        self.data = torch.tensor(raw_data, dtype=torch.float32)
        self.labels = torch.tensor(raw_labels, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# --- 3. ЦИКЛ ОБУЧЕНИЯ МОДЕЛИ ---
def train_model():
    # Гиперпараметры
    SEQ_LENGTH = 30   # длина скользящего окна (кол-во кадров для анализа)
    INPUT_SIZE = 66   # 33 точки * 2 координаты (x, y)
    BATCH_SIZE = 16
    EPOCHS = 20
    LR = 0.001
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Инициализация датасета и загрузчика
    dataset = FallVideoDataset(video_dir='videos/fall/', num_videos=13, seq_length=SEQ_LENGTH)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    # Инициализация сети
    model = FallDetectionLSTM(input_size=INPUT_SIZE).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    
    print(f"Начало обучения на устройстве: {DEVICE}")
    model.train()
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
            
        avg_loss = epoch_loss / total
        accuracy = correct / total
        print(f"Эпоха [{epoch+1}/{EPOCHS}] | Лосс: {avg_loss:.4f} | Точность: {accuracy*100:.2f}%")
        
    # Сохранение весов
    torch.save(model.state_dict(), 'fall_detection_lstm.pth')
    print("Веса модели успешно сохранены в 'fall_detection_lstm.pth'")

# --- 4. ДЕМО-ЗАПУСК / ДЕТЕКЦИЯ В РЕАЛЬНОМ ВРЕМЕНИ С LSTM ---
def run_inference_live(video_path='videos/fall/2.mp4'):
    SEQ_LENGTH = 30
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Загружаем обученную модель
    model = FallDetectionLSTM(input_size=66)
    if os.path.exists('fall_detection_lstm.pth'):
        model.load_state_dict(torch.load('fall_detection_lstm.pth', map_location=DEVICE))
        print("Веса модели загружены для инференса.")
    else:
        print("Файл весов не найден! Запустите сначала train_model().")
        return
        
    model.to(DEVICE)
    model.eval()
    
    # Конвейер MediaPipe
    model_path = 'mediapipe_weights/pose_landmarker_full.task'
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=model_path),
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.7,
        num_poses=1
    )
    
    cap = cv2.VideoCapture(video_path)
    h, w = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    
    # Буфер для хранения временной последовательности кадров
    input_buffer = []
    current_state = "Normal"
    
    BODY_CONNECTIONS = [
        (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
        (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
        (12, 24), (24, 26), (26, 28), (28, 30), (30, 32), (32, 28),
        (11, 23), (23, 25), (25, 27), (27, 29), (29, 31), (31, 27)
    ]
    
    frame_idx = 0
    with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break
                
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            timestamp_ms = int(frame_idx * (1000 / cap.get(cv2.CAP_PROP_FPS)))
            frame_idx += 1
            
            result = landmarker.detect_for_video(mp_image, timestamp_ms)
            
            frame_features = [0.0] * 66
            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                landmarks = result.pose_landmarks[0]
                frame_features = []
                for lm in landmarks:
                    frame_features.extend([lm.x, lm.y])
                
                # Отрисовка скелета
                for connection in BODY_CONNECTIONS:
                    start_idx, end_idx = connection
                    pt1, pt2 = landmarks[start_idx], landmarks[end_idx]
                    if pt1.visibility > 0.5 and pt2.visibility > 0.5:
                        cv2.line(frame, (int(pt1.x * w), int(pt1.y * h)), (int(pt2.x * w), int(pt2.y * h)), (0, 255, 0), 2)
            
            # Добавляем фичи кадра в буфер LSTM
            input_buffer.append(frame_features)
            if len(input_buffer) > SEQ_LENGTH:
                input_buffer.pop(0)
                
            # Если буфер заполнен — делаем предсказание LSTM
            if len(input_buffer) == SEQ_LENGTH:
                seq_tensor = torch.tensor([input_buffer], dtype=torch.float32).to(DEVICE)
                with torch.no_grad():
                    output = model(seq_tensor)
                    _, pred = torch.max(output, 1)
                    current_state = "FALL DETECTED!" if pred.item() == 1 else "Normal"# Отрисовка текста статуса на экране
                    color = (0, 0, 255) if current_state == "FALL DETECTED!" else (0, 255, 0)
                    cv2.putText(frame, f"STATE: {current_state}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2, cv2.LINE_AA)
                    resized_frame = cv2.resize(frame, (540, 960))
                    cv2.imshow('Mediapipe + LSTM Fall Detection', resized_frame)
                    if cv2.waitKey(10) & 0xFF == ord('q'):
                        break
    cap.release()
    cv2.destroyAllWindows()
                        
train_model()# 2. Демонстрация работы на примере видео номер 2
run_inference_live(video_path='videos/fall/2.mp4')



-> Начало извлечения ключевых точек из видео...
Файл не найден: videos/fall/13.mp4, пропускаем.
-> Извлечение завершено. Создано 2154 тренировочных последовательностей.
Начало обучения на устройстве: cpu
Эпоха [1/20] | Лосс: 0.4860 | Точность: 78.13%
Эпоха [2/20] | Лосс: 0.4099 | Точность: 84.45%
Эпоха [3/20] | Лосс: 0.3776 | Точность: 86.35%
Эпоха [4/20] | Лосс: 0.3595 | Точность: 86.26%
Эпоха [5/20] | Лосс: 0.3413 | Точность: 86.17%
Эпоха [6/20] | Лосс: 0.3650 | Точность: 86.07%
Эпоха [7/20] | Лосс: 0.3164 | Точность: 86.86%
Эпоха [8/20] | Лосс: 0.2800 | Точность: 87.93%
Эпоха [9/20] | Лосс: 0.2624 | Точность: 88.81%
Эпоха [10/20] | Лосс: 0.2579 | Точность: 89.42%
Эпоха [11/20] | Лосс: 0.2227 | Точность: 91.09%
Эпоха [12/20] | Лосс: 0.2568 | Точность: 88.58%
Эпоха [13/20] | Лосс: 0.2045 | Точность: 91.46%
Эпоха [14/20] | Лосс: 0.1958 | Точность: 91.92%
Эпоха [15/20] | Лосс: 0.1619 | Точность: 93.55%
Эпоха [16/20] | Лосс: 0.1776 | Точность: 92.76%
Эпоха [17/20] | Лосс: 0.1603 | Точнос

In [5]:
import cv2
import mediapipe as mp
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os

# --- 1. АРХИТЕКТУРА LSTM МОДЕЛИ ---
class FallDetectionLSTM(nn.Module):
    def __init__(self, input_size=66, hidden_size=64, num_layers=2, num_classes=2):
        super(FallDetectionLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM слой обрабатывает последовательность кадров
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        # Полносвязный слой для классификации состояния на текущем/последнем кадре
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        # Берем выход последнего временного шага для классификации
        out = self.fc(out[:, -1, :])
        return out

# --- 2. КАСТОМНЫЙ ДАТАСЕТ ДЛЯ ПРЕДОБРАБОТКИ ВИДЕО ---
class FallVideoDataset(Dataset):
    def __init__(self, video_dir, num_videos_fall=13, num_videos_normal=6, seq_length=30):
        self.seq_length = seq_length
        raw_data = []
        raw_labels = []
        
        # Настройка MediaPipe
        model_path = 'mediapipe_weights/pose_landmarker_full.task'
        options = mp.tasks.vision.PoseLandmarkerOptions(
            base_options=mp.tasks.BaseOptions(model_asset_path=model_path),
            running_mode=mp.tasks.vision.RunningMode.VIDEO,
            min_pose_detection_confidence=0.5,
            num_poses=1
        )
        
        print("Extracting FALL key points")
        
        for i in range(1, num_videos_fall + 1):
            video_path = os.path.join(video_dir+'fall/', f"{i}.mp4")  
            if not os.path.exists(video_path):
                print(f"Not found: {video_path}")
                continue
                
            cap = cv2.VideoCapture(video_path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            if fps <= 0: 
                fps = 30.0
                
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            
            landmarks_sequence = []
            frame_idx = 0
            
            with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
                while cap.isOpened():
                    success, frame = cap.read()
                    if not success:
                        break
                        
                    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                    timestamp_ms = int(frame_idx * (1000 / fps))
                    
                    try:
                        result = landmarker.detect_for_video(mp_image, timestamp_ms)
                        
                        # Вектор признаков текущего кадра
                        frame_features = []
                        if result.pose_landmarks and len(result.pose_landmarks) > 0:
                            # Извлекаем x, y для всех 33 точек первой найденной позы
                            center_x = result.pose_landmarks[0][23].x
                            center_y = result.pose_landmarks[0][23].y
                            for lm in result.pose_landmarks[0]:
                                frame_features.extend([lm.x-center_x, lm.y-center_y])
                        
                        # Если точки не найдены или их меньше 33, заполняем нулями до 66 признаков
                        if len(frame_features) < 66:
                            frame_features = [0.0] * 66
                            
                        landmarks_sequence.append((frame_features, 1))
                        
                    except Exception as e:
                        print(f"Error on frame {frame_idx} in {i}: {e}")
                        
                    frame_idx += 1
                    
            cap.release()
            
            # Нарезаем кадры ТЕКУЩЕГО видео на скользящие окна
            for idx in range(len(landmarks_sequence) - seq_length + 1):
                window = landmarks_sequence[idx : idx + seq_length]
                
                # Разделяем признаки и метки внутри окна
                window_features = [item[0] for item in window] # Список из 30 векторов по 66 элементов
                window_label = window[-1][1]                   # Метка последнего кадра в окне
                
                raw_data.append(window_features)
                raw_labels.append(window_label)


        print("Extracting NORMAL key points")
        
        for i in range(1, num_videos_normal + 1):
            video_path = os.path.join(video_dir+'no_fall/', f"{i}.mp4")  
            if not os.path.exists(video_path):
                print(f"Not found: {video_path}")
                continue
                
            cap = cv2.VideoCapture(video_path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            if fps <= 0: 
                fps = 30.0
                
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            
            landmarks_sequence = []
            frame_idx = 0
            
            with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
                while cap.isOpened():
                    success, frame = cap.read()
                    if not success:
                        break
                        
                    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                    timestamp_ms = int(frame_idx * (1000 / fps))
                    
                    try:
                        result = landmarker.detect_for_video(mp_image, timestamp_ms)
                        
                        # Вектор признаков текущего кадра
                        frame_features = []
                        if result.pose_landmarks and len(result.pose_landmarks) > 0:
                            # Извлекаем x, y для всех 33 точек первой найденной позы
                            center_x = result.pose_landmarks[0][23].x
                            center_y = result.pose_landmarks[0][23].y
                            for lm in result.pose_landmarks[0]:
                                frame_features.extend([lm.x-center_x, lm.y-center_y])
                        
                        # Если точки не найдены или их меньше 33, заполняем нулями до 66 признаков
                        if len(frame_features) < 66:
                            frame_features = [0.0] * 66
                            
                        landmarks_sequence.append((frame_features, 0))
                        
                    except Exception as e:
                        print(f"Error on frame {frame_idx} in {i}: {e}")
                        
                    frame_idx += 1
                    
            cap.release()
            
            # Нарезаем кадры ТЕКУЩЕГО видео на скользящие окна
            for idx in range(len(landmarks_sequence) - seq_length + 1):
                window = landmarks_sequence[idx : idx + seq_length]
                
                # Разделяем признаки и метки внутри окна
                window_features = [item[0] for item in window] # Список из 30 векторов по 66 элементов
                window_label = window[-1][1]                   # Метка последнего кадра в окне
                
                raw_data.append(window_features)
                raw_labels.append(window_label)
        if len(raw_data) == 0:
            raise RuntimeError(f"Error: {video_dir}")
            
        print(f"Success. Train sequences: {len(raw_data)}")
        
        # Преобразуем в тензоры PyTorch и сохраняем как атрибуты класса
        self.data = torch.tensor(raw_data, dtype=torch.float32)
        self.labels = torch.tensor(raw_labels, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# --- 3. ЦИКЛ ОБУЧЕНИЯ МОДЕЛИ ---


In [ ]:
def train_model():
    # Гиперпараметры
    SEQ_LENGTH = 30   # длина скользящего окна (кол-во кадров для анализа)
    INPUT_SIZE = 66   # 33 точки * 2 координаты (x, y)
    BATCH_SIZE = 16
    EPOCHS = 20
    LR = 0.001
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Инициализация датасета и загрузчика
    dataset = FallVideoDataset(video_dir='videos/', num_videos_fall=13, num_videos_normal=6, seq_length=SEQ_LENGTH)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    # Инициализация сети
    model = FallDetectionLSTM(input_size=INPUT_SIZE).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    
    print(f"Начало обучения на устройстве: {DEVICE}")
    model.train()
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
            
        avg_loss = epoch_loss / total
        accuracy = correct / total
        print(f"Эпоха [{epoch+1}/{EPOCHS}] | Лосс: {avg_loss:.4f} | Точность: {accuracy*100:.2f}%")
        
    # Сохранение весов
    torch.save(model.state_dict(), 'fall_detection_lstm.pth')
    print("Веса модели успешно сохранены в 'fall_detection_lstm.pth'")

# --- 4. ДЕМО-ЗАПУСК / ДЕТЕКЦИЯ В РЕАЛЬНОМ ВРЕМЕНИ С LSTM ---

train_model()# 2. Демонстрация работы на примере видео номер 2

In [83]:
import cv2
import mediapipe as mp
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os

null_vector = [0.0] * 66

        
def run_inference_live(video_path):
    SEQ_LENGTH = 15
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model = FallDetectionLSTM(input_size=66)
    if os.path.exists('fall_detection_lstm.pth'):
        model.load_state_dict(torch.load('fall_detection_lstm.pth', map_location=DEVICE))
        print("Model weights are loaded successfully")
    else:
        print("Missing fall_detection_lstm.pth")
        return
        
    model.to(DEVICE)
    model.eval()
    
    model_path = 'mediapipe_weights/pose_landmarker_full.task'
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=model_path),
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.8,
        num_poses=1
    )
    
    cap = cv2.VideoCapture(video_path)
    h, w = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    
    input_buffer = []
    current_state = "Normal"
    BODY_CONNECTIONS = [
        (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
        (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
        (12, 24), (24, 26), (26, 28), (28, 30), (30, 32), (32, 28),
        (11, 23), (23, 25), (25, 27), (27, 29), (29, 31), (31, 27)
    ]
    
    frame_idx = 0
    if h==w==-1:
        print("Failed to open the video")
        return
        
    current_state = "NO PERSON" 
    with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break
                
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            timestamp_ms = int(frame_idx * (1000 / cap.get(cv2.CAP_PROP_FPS)))
            frame_idx += 1
            
            result = landmarker.detect_for_video(mp_image, timestamp_ms)
            
            frame_features = null_vector
            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                landmarks = result.pose_landmarks[0]
                frame_features = []
                center_x = landmarks[23].x
                center_y = landmarks[23].y
                for lm in landmarks:
                    frame_features.extend([lm.x-center_x, lm.y-center_y])
                
                for connection in BODY_CONNECTIONS:
                    start_idx, end_idx = connection
                    pt1, pt2 = landmarks[start_idx], landmarks[end_idx]
                    if pt1.visibility > 0.5 and pt2.visibility > 0.5:
                        cv2.line(frame, (int(pt1.x * w), int(pt1.y * h)), (int(pt2.x * w), int(pt2.y * h)), (0, 255, 0), 2)
            
            input_buffer.append(frame_features)
            if len(input_buffer) > SEQ_LENGTH:
                input_buffer.pop(0)
                
            
            if len(input_buffer) == SEQ_LENGTH:
                seq_tensor = torch.tensor([input_buffer], dtype=torch.float32).to(DEVICE)
                with torch.no_grad():
                    output = model(seq_tensor)
                    _, pred = torch.max(output, 1)

                    
                    
                    if null_vector == input_buffer[-1]:
                        current_state = "NO PERSON"
                    elif pred.item() == 1 and not(input_buffer[0] == null_vector):
                        current_state = "FALL DETECTED!"    
                    else:
                        current_state = "NORMAL"

                    
                    if current_state == "FALL DETECTED!":
                        color = (0, 0, 255)
                    elif current_state == "NO PERSON":
                        color = (255, 0, 0)
                    else:
                        color =(0, 255, 0)
                    cv2.putText(frame, f"STATE: {current_state}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2, cv2.LINE_AA)
                    resized_frame = cv2.resize(frame, (540, 960))
                    cv2.imshow('Mediapipe + LSTM Fall Detection', resized_frame)
                    if cv2.waitKey(10) & 0xFF == ord('q'):
                        break
    cap.release()
    cv2.destroyAllWindows()
                        




In [90]:
run_inference_live(video_path='videos/fall/5_n.mp4')
#run_inference_live(video_path=0)

Model weights are loaded successfully


In [20]:
print('wdsfds')

wdsfds
